In [1]:
print("WEB Scraping Dailymirror.lk Breaking news")

WEB Scraping Dailymirror.lk Breaking news


In [4]:
import cloudscraper
import requests
from bs4 import BeautifulSoup
import mysql.connector
import time
import random
import html

### Connect the database and i create the database named "news_data"

## Table Creating One time 

In [12]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS scraped_news (
    id INT AUTO_INCREMENT PRIMARY KEY,
    title TEXT,
    content LONGTEXT,
    date VARCHAR(50),
    link VARCHAR(250) UNIQUE
)
""")
db.commit()

print("Table created successfully")

Table created successfully


## Defining the URL and extract information form the web side 

In [13]:

BASE_URL = "https://www.dailymirror.lk/latest-news/108"
TOTAL_PAGES = 1000
STEP = 30

scraper = cloudscraper.create_scraper()


def is_duplicate(link):
    cursor.execute("SELECT id FROM scraped_news WHERE link=%s", (link,))
    return cursor.fetchone() is not None


def clean_text(text):
    if not text:
        return "N/A"

    text = html.unescape(text)

    text = text.replace("â€™", "’") \
               .replace("â€œ", "“") \
               .replace("â€\x9d", "”") \
               .replace("â€“", "–") \
               .replace("â€˜", "‘")
    return text


def scrape_page(url):
    try:
        response = scraper.get(url, timeout=20)
        print("Status Code:", response.status_code)

        if response.status_code != 200:
            print(f" Failed: {url}")
            return 0


        soup = BeautifulSoup(response.content, "html.parser", from_encoding='utf-8')

        news_items = soup.find_all("div", class_="col-md-8")
        inserted = 0

        for item in news_items:

            title_tag = item.find("h3", class_="cat_title")
            title = clean_text(title_tag.get_text(strip=True)) if title_tag else "N/A"


            content_tag = item.find("p", class_="text-dark")
            content = clean_text(content_tag.get_text(strip=True)) if content_tag else "N/A"


            date_tag = item.find("h4", class_="text-secondary")
            date = clean_text(date_tag.get_text(strip=True)) if date_tag else "N/A"


            link = "N/A"
            if title_tag and title_tag.find_parent("a"):
                link = title_tag.find_parent("a").get("href")


            if link != "N/A" and not is_duplicate(link):
                try:
                    cursor.execute("""
                        INSERT INTO scraped_news (title, content, date, link)
                        VALUES (%s, %s, %s, %s)
                    """, (title, content, date, link))
                    db.commit()
                    inserted += 1
                except mysql.connector.Error as e:
                    print("Insert error:", e)

        return inserted

    except Exception as e:
        print("Error:", e)
        return 0


def main():
    total_inserted = 0

    for page in range(TOTAL_PAGES):
        offset = page * STEP
        if offset == 0:
            url = BASE_URL
        else:
            url = f"{BASE_URL}/{offset}"

        print(f"\nScraping page {page+1}/{TOTAL_PAGES} ... {url}")
        inserted = scrape_page(url)
        total_inserted += inserted

        print(f"Inserted {inserted} records...")
        print(f"Total inserted so far: {total_inserted}")


        time.sleep(random.uniform(2, 5))

    print(" Scraping Completed Successfully!")
    print("Total inserted:", total_inserted)

    cursor.close()
    db.close()

if __name__ == "__main__":
    main()


Scraping page 1/1000 ... https://www.dailymirror.lk/latest-news/108
Status Code: 200
Inserted 30 records...
Total inserted so far: 30

Scraping page 2/1000 ... https://www.dailymirror.lk/latest-news/108/30
Status Code: 200
Inserted 30 records...
Total inserted so far: 60

Scraping page 3/1000 ... https://www.dailymirror.lk/latest-news/108/60
Status Code: 200
Inserted 30 records...
Total inserted so far: 90

Scraping page 4/1000 ... https://www.dailymirror.lk/latest-news/108/90
Status Code: 200
Inserted 30 records...
Total inserted so far: 120

Scraping page 5/1000 ... https://www.dailymirror.lk/latest-news/108/120
Status Code: 200
Inserted 30 records...
Total inserted so far: 150

Scraping page 6/1000 ... https://www.dailymirror.lk/latest-news/108/150
Status Code: 200
Inserted 30 records...
Total inserted so far: 180

Scraping page 7/1000 ... https://www.dailymirror.lk/latest-news/108/180
Status Code: 200
Inserted 30 records...
Total inserted so far: 210

Scraping page 8/1000 ... http